# Categorical Jacobians

Compute categorical Jacobians from ESM logits for each haplotype.

Adapted from: https://colab.research.google.com/github/sokrypton/ColabBio/blob/main/categorical_jacobian/esm2.ipynb#scrollTo=DtRKmskxgHfs

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True


import src.utils as utils
import src.config as config
import src.haplosaurus as hs
import src.vep_pipeline as vp
import src.vep_analysis as va
import src.vep_metrics as vm
import src.proteingym as pg 
import src.onekg as og

import matplotlib.pyplot as plt
import seaborn as sns
import pooch
from tqdm import tqdm
import polars as pl
import glob

import src.ColabFold.categorical_jacobians as cj
import bokeh
bokeh.io.output_notebook()

pd.set_option('display.max_columns', None)

/home/schilder/.conda/envs/esm2/lib/python3.12/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


Loading BokehJS ...

## Load haplotype sequences

In [ ]:
from Bio import SeqIO

fasta_file = os.path.expanduser("~/projects/data/colabfold/ENST00000357654/ENST00000357654.fasta")
haplotype_seqs = list(SeqIO.parse(fasta_file, "fasta"))

ENSP00000350283:REF
MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLQESTRFSQLVEELLKIICAFQLDTGLEYANSYNFAKKENNSPEHLKDEVSIIQSMGYRNRAKRLLQSEPENPSLQETSLSVQLSNLGTVRTLRTKQRIQPQKTSVYIELGSDSSEDTVNKATYCSVGDQELLQITPQGTRDEISLDSAKKAACEFSETDVTNTEHHQPSNNDLNTTEKRAAERHPEKYQGSSVSNLHVEPCGTNTHASSLQHENSSLLLTKDRMNVEKAEFCNKSKQPGLARSQHNRWAGSKETCNDRRTPSTEKKVDLNADPLCERKEWNKQKLPCSENPRDTEDVPWITLNSSIQKVNEWFSRSDELLGSDDSHDGESESNAKVADVLDVLNEVDEYSGSSEKIDLLASDPHEALICKSERVHSKSVESNIEDKIFGKTYRKKASLPNLSHVTENLIIGAFVTEPQIIQERPLTNKLKRKRRPTSGLHPEDFIKKADLAVQKTPEMINQGTNQTEQNGQVMNITNSGHENKTKGDSIQNEKNPNPIESLEKESAFKTKAEPISSSISNMELELNIHNSKAPKKNRLRRKSSTRHIHALELVVSRNLSPPNCTELQIDSCSSSEEIKKKKYNQMPVRHSRNLQLMEGKEPATGAKKSNKPNEQTSKRHDSDTFPELKLTNAPGSFTKCSNTSELKEFVNPSLPREEKEEKLETVKVSNNAEDPKDLMLSGERVLQTERSVESSSISLVPGTDYGTQESISLLEVSTLGKAKTEPNKCVSQCAAFENPKGLIHGCSKDNRNDTEGFKYPLGHEVNHSRETSIEMEESELDAQYLQNTFKVSKRQSFAPFSNPGNAEEECATFSAHSGSLKKQSPKVTFECEQKEENQGKNESNIKPVQTVNITAGFPVVGQKDKPVDNAKCSIKGGSRFCLSSQFRGNETGLITPNKHGLLQNPYRI

In [32]:
sequence = cj.process_sequence(haplotype_seqs[0].seq)
sequence

'MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLQESTRFSQLVEELLKIICAFQLDTGLEYANSYNFAKKENNSPEHLKDEVSIIQSMGYRNRAKRLLQSEPENPSLQETSLSVQLSNLGTVRTLRTKQRIQPQKTSVYIELGSDSSEDTVNKATYCSVGDQELLQITPQGTRDEISLDSAKKAACEFSETDVTNTEHHQPSNNDLNTTEKRAAERHPEKYQGSSVSNLHVEPCGTNTHASSLQHENSSLLLTKDRMNVEKAEFCNKSKQPGLARSQHNRWAGSKETCNDRRTPSTEKKVDLNADPLCERKEWNKQKLPCSENPRDTEDVPWITLNSSIQKVNEWFSRSDELLGSDDSHDGESESNAKVADVLDVLNEVDEYSGSSEKIDLLASDPHEALICKSERVHSKSVESNIEDKIFGKTYRKKASLPNLSHVTENLIIGAFVTEPQIIQERPLTNKLKRKRRPTSGLHPEDFIKKADLAVQKTPEMINQGTNQTEQNGQVMNITNSGHENKTKGDSIQNEKNPNPIESLEKESAFKTKAEPISSSISNMELELNIHNSKAPKKNRLRRKSSTRHIHALELVVSRNLSPPNCTELQIDSCSSSEEIKKKKYNQMPVRHSRNLQLMEGKEPATGAKKSNKPNEQTSKRHDSDTFPELKLTNAPGSFTKCSNTSELKEFVNPSLPREEKEEKLETVKVSNNAEDPKDLMLSGERVLQTERSVESSSISLVPGTDYGTQESISLLEVSTLGKAKTEPNKCVSQCAAFENPKGLIHGCSKDNRNDTEGFKYPLGHEVNHSRETSIEMEESELDAQYLQNTFKVSKRQSFAPFSNPGNAEEECATFSAHSGSLKKQSPKVTFECEQKEENQGKNESNIKPVQTVNITAGFPVVGQKDKPVDNAKCSIKGGSRFCLSSQFRGNETGLITPNKHGLLQNPYRIPPLFPIKSFVKTKCKKNLL

## Load model

In [ ]:
model_name = "esm2_t33_650M_UR50D"
model, alphabet = cj.load_model(model_name = model_name)  

## Compute logits

In [38]:
ALPHABET_map = cj.get_alphabet_map(alphabet)
logits, jac = cj.get_logits(seq=sequence, alphabet=alphabet, model=model, return_jac=True)
logits = logits[:,ALPHABET_map]  

Computing logits and jacobians


  0%|          | 0/1863 [elapsed: 00:00 remaining: ?]

Logits computed in 1484.07 seconds


### Plot

In [39]:
cj.plot_logits(logits=logits,  
               save_path=f"output/{model_name}/conservation_logits.txt"
               )

[autoreload of numpy._core._add_newdocs failed: Traceback (most recent call last):
  File "/home/schilder/.conda/envs/esm2/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 283, in check
    superreload(m, reload, self.old_objects)
  File "/home/schilder/.conda/envs/esm2/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 483, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/home/schilder/.conda/envs/esm2/lib/python3.12/importlib/__init__.py", line 131, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 866, in _exec
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/home/schilder/.conda/envs/esm2/lib/python3.12/site-packages/numpy/_core/_add_newdocs.py", line 4445, in <module>
    add_newdoc('numpy._core.multiarray', 'ndarray', ('tostring', r"""
  File "/home/schilder/.co

AttributeError: module 'numpy.lib' has no attribute '_datasource'

## Compute Categorical Jacobians

In [37]:
fast=True
if fast:
    logits, jac = cj.get_logits(seq=sequence, alphabet=alphabet, model=model, return_jac=True)
else:
    jac = cj.get_categorical_jacobian(seq=sequence, alphabet=alphabet, model=model, fast=fast)

logits = logits[:,ALPHABET_map] 

cj.plot_logits(logits=logits,  
               save_path=f"output/{model_name}/conservation_logits.txt"
               )

Computing categorical jacobians


  0%|          | 0/1863 [elapsed: 00:00 remaining: ?]

KeyboardInterrupt: 

### Plot

In [ ]:
con = cj.plot_jac(jac=jac,
                    ALPHABET_map=ALPHABET_map,
                    contacts_save_path=f"output/{model_name}/coevolution.txt",
                    jac_save_path=f"output/{model_name}/jac.npy")

### Show top covarying positions

In [ ]:
from google.colab import data_table

con_df = cj.contact_to_dataframe(con["contacts"]) 
sub_df = con_df[con_df["j"]>con_df["i"]].sort_values('value',ascending=False)
data_table.DataTable(sub_df, include_index=False, num_rows_per_page=20, min_width=10)

#### Select pairs of residues to investigate

In [ ]:
from bokeh.models import BasicTicker, PrintfTickFormatter
from bokeh.palettes import viridis, RdBu
from bokeh.transform import linear_cmap
from bokeh.plotting import figure, show
position_i = 15 # @param {type:"integer"}
position_j = 57 # @param {type:"integer"}

i = position_i - 1
j = position_j - 1
df = cj.pair_to_dataframe(con["jac"][i,:,j,:], cj.ALPHABET)

# plot pssm
TOOLS = "hover,save,pan,box_zoom,reset,wheel_zoom"
p = figure(title=f"coevolution between {position_i} {position_j}",
            x_range=list(cj.ALPHABET),
            y_range=list(cj.ALPHABET)[::-1],
            width=400, height=400,
            tools=TOOLS, toolbar_location='below',
            tooltips=[('aa_i', '@aa_i'), ('aa_j', '@aa_j'), ('value', '@value')])
p.xaxis.axis_label = f"{sequence[i]}{position_i}"
p.yaxis.axis_label = f"{sequence[j]}{position_j}"

r = p.rect(x="aa_i", y="aa_j", width=1, height=1, source=df,
            fill_color=linear_cmap('value', cj.bwr_r, low=-3.0, high=3.0),
            line_color=None, dilate=True)
show(p)
